In [ ]:
%pip install "rfdetr[train,loggers,onnx]==1.7.1" -q
%pip install supervision roboflow -q

In [ ]:
import os
HOME = os.getcwd()
print(HOME)

In [ ]:
# Allow access to personal google drive and add new folders

# Connect Google Drive
from google.colab import drive
drive.mount("/content/drive", force_remount=True) # This will prompt for authorization.

# This will create the uavs files if they don't exist.
folders =  ["uavs-rfdetr/"]
for folder in folders:
  path = "/content/drive/MyDrive/" + folder
  if not os.path.exists(path): # Create the folder if it does not exist
    os.mkdir(path)

In [ ]:
!mkdir {HOME}/datasets
%cd {HOME}/datasets
from google.colab import userdata
uavs_folder = "/content/drive/MyDrive/uavs-rfdetr/"

!mkdir -p /content/datasets/savedir/
!cp -r "/content/drive/MyDrive/uavs-rfdetr/images.tar.gz" "/content/datasets/savedir/"

!cp -r "/content/drive/MyDrive/uavs-rfdetr/labels.tar.gz" "/content/datasets/savedir/"

!tar xf /content/datasets/savedir/images.tar.gz --directory /content/datasets/savedir/

!tar xf /content/datasets/savedir/labels.tar.gz --directory /content/datasets/savedir/

In [ ]:
## create directories
!mkdir /content/datasets/train/
!mkdir /content/datasets/train/images/
!mkdir /content/datasets/train/labels/
!mkdir /content/datasets/test/
!mkdir /content/datasets/test/images/
!mkdir /content/datasets/test/labels/
!mkdir /content/datasets/valid/
!mkdir /content/datasets/valid/images/
!mkdir /content/datasets/valid/labels/

#get the data.yaml file
!cp "/content/drive/MyDrive/uavs-rfdetr/data.yaml" "/content/datasets/data.yaml"
!ls /content/datasets/

#move the data to the expected directories
!cp -r "/content/datasets/savedir/images/train/." "/content/datasets/train/images/"
!cp -r "/content/datasets/savedir/labels/train/." "/content/datasets/train/labels/"

!cp -r "/content/datasets/savedir/images/test/." "/content/datasets/test/images/"
!cp -r "/content/datasets/savedir/labels/test/." "/content/datasets/test/labels/"

!cp -r "/content/datasets/savedir/images/val/." "/content/datasets/valid/images/"
!cp -r "/content/datasets/savedir/labels/val/." "/content/datasets/valid/labels/"

!ls /content/datasets/

In [ ]:
# check the number of images and labels in each folder
for split in ["train", "valid", "test"]:
    imgs = len(os.listdir(f"/content/datasets/{split}/images"))
    labels = len(os.listdir(f"/content/datasets/{split}/labels"))
    print(split, imgs, labels)

In [ ]:
from rfdetr import RFDETRNano

model = RFDETRNano()

model.train(
    dataset_dir="/content/datasets/",
    epochs=20,
    batch_size=16,
    grad_accum_steps=1,
    output_dir = "/content/outputs"
)

In [ ]:
#If Colab restarts, restore from Drive first:
!cp -r /content/drive/MyDrive/uavs-rfdetr/outputs /content/outputs

In [ ]:
from rfdetr import RFDETRNano

# resume training from checkpoint
model = RFDETRNano()

model.train(
    dataset_dir="/content/datasets/",
    epochs=30,           # total epochs, not how many more
    batch_size=16,
    grad_accum_steps=1,
    output_dir="/content/outputs",
    resume="/content/outputs/last.ckpt"   # <-- resume here
)

In [ ]:
!cp -r "/content/outputs" "/content/drive/MyDrive/uavs-rfdetr/"

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("/content/outputs/metrics.csv")

# Train and val metrics are on separate rows — split them out
val_df = df[df["val/mAP_50"].notna()].copy()
train_df = df[df["train/loss"].notna()].copy()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# mAP
axes[0].plot(val_df["epoch"], val_df["val/mAP_50"], marker="o", label="mAP@50")
axes[0].plot(val_df["epoch"], val_df["val/ema_mAP_50"], marker="s", linestyle="--", label="EMA mAP@50")
axes[0].plot(val_df["epoch"], val_df["val/mAP_50_95"], marker="o", label="mAP@50:95")
axes[0].set_title("mAP"); axes[0].set_xlabel("Epoch"); axes[0].legend(); axes[0].grid(True)

# Precision / Recall / F1
axes[1].plot(val_df["epoch"], val_df["val/precision"], marker="o", label="Precision")
axes[1].plot(val_df["epoch"], val_df["val/recall"], marker="o", label="Recall")
axes[1].plot(val_df["epoch"], val_df["val/F1"], marker="o", label="F1")
axes[1].set_title("Precision / Recall / F1"); axes[1].set_xlabel("Epoch"); axes[1].legend(); axes[1].grid(True)

# Loss
axes[2].plot(train_df["epoch"], train_df["train/loss"], marker="o", label="Train loss")
axes[2].plot(val_df["epoch"], val_df["val/loss"], marker="s", linestyle="--", label="Val loss")
axes[2].set_title("Loss"); axes[2].set_xlabel("Epoch"); axes[2].legend(); axes[2].grid(True)

plt.tight_layout()
plt.savefig("/content/drive/MyDrive/uavs-rfdetr/metrics.png", dpi=150)
plt.show()

In [ ]:
from rfdetr import RFDETRNano

model = RFDETRNano(pretrain_weights="/content/outputs/checkpoint_best_total.pth")

# export best checkpoint to onnx
model.export(
    format="onnx",
    output_dir="/content/drive/MyDrive/uavs-rfdetr/onnx_export"
)


In [ ]:
from rfdetr import RFDETRNano

model = RFDETRNano(pretrain_weights="/content/outputs/checkpoint_best_ema.pth")

# export ema checkpoint to onnx
model.export(
    format="onnx",
    output_dir="/content/drive/MyDrive/uavs-rfdetr/onnx_export"
)